# 04 · Do stages share structure? Compositional reuse in EMG and pose

**Investigation ladder** (see [`EVALUATION.md`](../EVALUATION.md)):
- **Stage 3 · Sharing / composition: do stages reuse shared synergies?**

**Read first:** Lazzari & Saxena 2025, *Multitasking recurrent networks utilize compositional strategies for control of movement* (bioRxiv 10.1101/2025.09.10.675375); Tresch, Cheung & d'Avella 2006 (synergy extraction)

**We expect:** the 15 stages reuse a shared low-dimensional EMG space (far above random subspaces), and synergies learned on instructed stages explain free-style movement nearly as well as the free-style stage's own. Following Lazzari & Saxena, EMG (muscle) structure should be **less** shared across stages than pose structure. **Wrong if:** other stages' subspaces capture no more of a stage than random subspaces do, or instructed synergies explain free-style EMG no better than random synergies.

**The idea from the paper, and how we borrow it.** Lazzari & Saxena trained one recurrent network to make ten
different arm movements through a simulated 6-muscle arm. The network **reused** pieces across tasks: tasks shared a
low-dimensional activity space, and a new task could be learned by changing only its input. They also found that the
*muscle* activity was **less** shared across tasks than the network's activity. Their result comes from a
**simulated** network and arm. Here we ask the same questions of **real** EMG and hand pose from **one person**: do the
15 stages (pinches, counting, pointing, free style …) reuse shared structure, and is that sharing stronger for EMG or
for pose?

**How this notebook works** (the same in every notebook)
- **`# TODO`** cells are yours. Each has a docstring with the input and output shapes. You write the core
  algorithms yourself in NumPy.
- Under every TODO there are three **💡 hints** that open one at a time: the plan in words, the tools with a tiny
  example, then a skeleton with blanks (`____`).
- After each TODO a **✅ check cell** compares your answer with the reference in [`ref04.py`](ref04.py), then a **📊 plot**
  shows what your code did. Other notebooks use the reference, so your blanks never break theirs.
- **✍️ questions** are for your own words, in the markdown cell under them.
- The **Result** cell at the end is the **gate**: it decides whether the next stage of the investigation is worth running.

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Proposal:** ➕ beyond the proposal, but it feeds Aim 1: shared structure across tasks is what makes a synergy bottleneck worth testing on new movements and new people.

**Goal:** measure how much the stages share their EMG and pose structure, and whether that sharing beats chance.

**Why:** if every stage used its own private muscle patterns, a few synergies could not serve as a general bottleneck. Stage 4 onwards assumes they can.

**Milestones**
1. Per-stage EMG and pose subspaces, and how many dimensions each stage needs
2. 15 × 15 cross-stage variance-explained ratios for EMG and pose, against random subspaces
3. Principal angles between stages, against random subspaces
4. A paired test, with a CI over independent blocks: is EMG more or less shared than pose?
5. Canonical correlations between synergy activations and pose scores, against a time-shift null
6. Instructed → free-style transfer of synergies

**Gate (decides the next stage):** the cross-stage sharing index (0 = random subspace, 1 = the stage's own subspace) is clearly above 0 (CI excludes 0) **and** instructed synergies explain free-style EMG nearly as well as its own (centred VAF ratio ≥ 0.9) and far better than random synergies.

**Done when:** every ✅ check passes and the gate table is filled in.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    else:
        !git -C {ROOT} pull -q                 # pick up files pushed since the first clone
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

_d = ROOT / "notebooks" / "experiments" / "04_compositional_structure"
assert (_d / "ref04.py").exists(), f"{_d}/ref04.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref04", None)
_d = ROOT / "notebooks" / "experiments" / "02_how_many_synergies"
assert (_d / "ref02.py").exists(), f"{_d}/ref02.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref02", None)
_d = ROOT / "notebooks" / "experiments" / "common"
sys.path.insert(0, str(_d)); sys.modules.pop("protocol", None)
import protocol                 # shared evaluation protocol: sessions, folds, baselines, nulls, CIs
import ref04 as ref          # the reference answers, used only by the ✅ check cells

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01's prep.py (data/ is never committed)

In [ ]:
# ── Load the shared data (builds it with prep.py the first time) ─────────────
import numpy as np
import matplotlib.pyplot as plt
import ref02                                   # NMF and PCA (notebook 02's reference; paste your own if you like)
sys.path.insert(0, str(ROOT / "notebooks" / "experiments" / "01_data_prep")); sys.modules.pop("prep", None)
import prep
if not prep.is_current(PROCESSED):   # missing, or built by an older version of prep.py
    subprocess.run([sys.executable, "notebooks/experiments/01_data_prep/prep.py"], check=True)
d = np.load(PROCESSED)

emg        = d["emg"].astype(np.float64)   # (N, 16)  EMG envelope, 50 samples/s, non-negative
pose       = d["pose"].astype(np.float64)  # (N, 20)  joint angles in degrees
pose_valid = d["pose_valid"]               # (N,)     False where pose tracking failed
rec        = d["rec"]                      # (N,)     which recording (0–29) each row comes from
split      = d["split"]                    # (N,)     0 = train, 1 = test, -1 = gap
stage_of_rec, filename = d["stage"], d["filename"]   # (30,) per recording

train, test = split == 0, split == 1
stage_per_frame = stage_of_rec[rec]        # (N,) str
STAGES = sorted(np.unique(stage_of_rec))   # 15 stage names, the row/column order of every matrix below
K = 4                                      # subspace size used for every stage and both signals (notebook 02's k)
SEED = 0

# Independent units. The two hands were recorded at the same time: each pair of simultaneous recordings is one
# BLOCK (15 blocks, paired by the recording number in the file name). Stages linked through a block form one group.
block_rec = protocol.block_of_rec(filename)                 # (30,) block of each recording
groups = protocol.stage_groups(stage_of_rec, filename)      # 14 groups of stages
group_of_stage = {s: g for g, (name, _) in enumerate(groups) for s in name.split(" + ")}
SIMUL = ref.simultaneous_pairs(STAGES, stage_of_rec, block_rec)   # (15, 15) different stages recorded together
print(f"{len(STAGES)} stages, {len(np.unique(block_rec))} blocks, {len(groups)} independent stage groups")
for i, j in zip(*np.where(np.triu(SIMUL))):
    print(f"  recorded at the same time (left out of cross-stage comparisons): {STAGES[i]} + {STAGES[j]}")

### The variables you start with
| Name | Shape | What it is |
|---|---|---|
| `emg`, `pose` | (N, 16), (N, 20) | EMG envelope; joint angles in degrees |
| `pose_valid`, `train`, `test` | (N,) bool | usable pose; first 80% / last 20% of every recording |
| `stage_per_frame` | (N,) str | the stage of every frame |
| `STAGES` | 15 str | stage names, sorted: the order of every 15 × 15 matrix |
| `K` | int | subspace size (4) for every stage and both signals |
| `groups`, `group_of_stage` | 14 groups | **independent units** for confidence intervals (see below) |
| `SIMUL` | (15, 15) bool | True for two different stages recorded at the same time (left out of cross-stage comparisons) |

**Why 14 groups, not 15 stages or 30 recordings?** The two hands were recorded **at the same time**, so each pair of
simultaneous recordings is one **block** (15 blocks), not two independent samples. Usually both hands of a block
have the same stage. In two blocks, though, one hand did *OneHandedFreeStyle* while the other, **idle**, hand's
recording is labelled *unconstrained*. Those two stages share blocks, so they count as **one** group (14 groups),
and the pair is left out of every stage-vs-stage comparison (`SIMUL`): their similarity could just come from being
recorded together. Confidence intervals resample the 14 groups.

**Fitting and scoring.** Every subspace is fitted on a stage's **train** frames (first 80%) and scored on its
**test** frames (last 20%). Then a stage's own subspace gets no unfair in-sample advantage over other stages'
subspaces.

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:*

---
## Task 1 · A subspace for every stage
For each stage, find the directions its data varies along most: its **top-d PCA subspace**. That is the stage's
"activity space" in the paper's language.

1. `dims_for_var(X, 0.9)`: how many principal components explain 90% of the (centred) variance. A descriptive
   number: does one stage need more dimensions than another?
2. `stage_basis(X, d)`: the top-`d` principal directions as **columns** of a (D, d) matrix `Q`. Each column has
   length 1 and the columns are at right angles to each other (*orthonormal*), so `X @ Q` gives the coordinates
   of every frame inside the subspace.

Both come from the SVD you used in notebook 02: centre `X`, `U, S, Vt = np.linalg.svd(...)`, the rows of `Vt` are
the directions and `S**2` is the variance along each one. Note that here the directions are **columns** (`Vt[:d].T`),
because the next tasks multiply `X @ Q`.

We use the **same d = K = 4** for every stage and for both EMG and pose. A bigger subspace always captures more,
so comparing stages (or EMG with pose) is only fair at a fixed size.

In [ ]:
def dims_for_var(X: np.ndarray, var_target: float = 0.9) -> int:
    """Smallest number of principal components explaining >= var_target of X's centred variance.

    Parameters
    ----------
    X          : (T, D) float
    var_target : float in (0, 1]

    Returns
    -------
    int between 1 and D
    """
    # TODO
    raise NotImplementedError


def stage_basis(X: np.ndarray, d: int) -> np.ndarray:
    """Top-d principal directions of X as orthonormal COLUMNS.

    Parameters
    ----------
    X : (T, D) float
    d : int

    Returns
    -------
    Q : (D, d) float, Q.T @ Q = identity, most variance first
    """
    # TODO
    raise NotImplementedError


# One subspace per stage, fitted on TRAIN frames; test frames kept for scoring
Q_emg:  dict[str, np.ndarray] = {}     # stage → (16, K)
Q_pose: dict[str, np.ndarray] = {}     # stage → (20, K)
X_emg:  dict[str, np.ndarray] = {}     # stage → (T_test, 16)   test EMG of that stage
X_pose: dict[str, np.ndarray] = {}     # stage → (T_test, 20)   test pose (valid frames only)
dims90 = {"EMG": [], "pose": []}       # per stage, in STAGES order

for s in STAGES:
    in_s = stage_per_frame == s
    # TODO: Q_emg[s] from emg[train & in_s]; Q_pose[s] from pose[train & in_s & pose_valid]   (stage_basis, d = K)
    # TODO: X_emg[s] = emg[test & in_s]; X_pose[s] = pose[test & in_s & pose_valid]
    # TODO: append dims_for_var of the TRAIN EMG and TRAIN pose of the stage to dims90["EMG"] and dims90["pose"]
    ...

#### 💡 Stuck on Task 1? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `dims_for_var`: centre, take the singular values `S`, turn `S**2` into a cumulative fraction of the total, and
   find the first position where it reaches the target. Add 1 (positions start at 0, counts at 1).
2. `stage_basis`: centre, SVD, keep the first `d` rows of `Vt` and transpose them into columns.
3. The loop: two masks per stage (train and test), the valid-pose mask for the pose only.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.svd(Xc, compute_uv=False)` returns only `S`; `full_matrices=False` keeps `Vt` small.
- `np.cumsum(a) / a.sum()` → cumulative fraction. `np.searchsorted(frac, 0.9)` → first index where `frac >= 0.9`.

```python
frac = np.cumsum(np.array([6.0, 3.0, 1.0])) / 10    # [0.6, 0.9, 1.0]
int(np.searchsorted(frac, 0.9) + 1)                  # 2 components reach 90%
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def dims_for_var(X, var_target=0.9):
    S = np.linalg.svd(X - X.mean(axis=0), compute_uv=False)
    frac = np.cumsum(S ** 2) / np.sum(____)
    return int(np.searchsorted(frac, ____) + 1)

def stage_basis(X, d):
    _, _, Vt = np.linalg.svd(X - X.mean(axis=0), full_matrices=False)
    return Vt[:____].T

for s in STAGES:
    in_s = stage_per_frame == s
    Q_emg[s]  = stage_basis(emg[train & in_s], K)
    Q_pose[s] = stage_basis(pose[train & in_s & ____], K)
    X_emg[s]  = emg[test & in_s]
    X_pose[s] = pose[test & in_s & pose_valid]
    dims90["EMG"].append(dims_for_var(emg[train & in_s]))
    dims90["pose"].append(dims_for_var(pose[____ & in_s & pose_valid]))
```

</details>

In [ ]:
# ✅ Check Task 1
rng_t = np.random.default_rng(0)
X_toy = rng_t.normal(size=(500, 6)) * np.array([5, 3, 1, 0.1, 0.1, 0.1])
assert dims_for_var(X_toy, 0.9) == ref.dims_for_var(X_toy, 0.9), \
    f"dims_for_var = {dims_for_var(X_toy, 0.9)}, expected {ref.dims_for_var(X_toy, 0.9)}"
Q_toy = stage_basis(X_toy, 3)
assert Q_toy.shape == (6, 3), f"stage_basis shape {Q_toy.shape}, expected (6, 3): columns, not rows"
assert np.allclose(Q_toy.T @ Q_toy, np.eye(3), atol=1e-8), "columns must be orthonormal"
assert np.allclose(np.abs(Q_toy.T @ ref.stage_basis(X_toy, 3)), np.eye(3), atol=1e-6), "wrong directions"
assert len(Q_emg) == len(Q_pose) == 15 and Q_emg[STAGES[0]].shape == (16, K) and Q_pose[STAGES[0]].shape == (20, K)
s0 = STAGES[0]
assert np.allclose(np.abs(Q_emg[s0].T @ ref.stage_basis(emg[train & (stage_per_frame == s0)], K)), np.eye(K), atol=1e-6), \
    "Q_emg should be fitted on the TRAIN frames of each stage"
assert len(X_pose[s0]) == (test & (stage_per_frame == s0) & pose_valid).sum(), "X_pose: test frames with valid pose"
print("✅ 15 stage subspaces for EMG and for pose")

In [ ]:
# 📊 How many dimensions does each stage need for 90% of its variance?
fig, ax = plt.subplots(figsize=(12, 3.5))
x = np.arange(len(STAGES))
ax.bar(x - 0.2, dims90["EMG"], 0.4, label="EMG (of 16)")
ax.bar(x + 0.2, dims90["pose"], 0.4, label="pose (of 20)")
ax.set_xticks(x, [s[:18] for s in STAGES], rotation=60, ha="right", fontsize=8)
ax.set_ylabel("PCs for 90% variance"); ax.legend(); plt.tight_layout(); plt.show()

*What to look for:* do free-style stages (`…Free…`, `unconstrained`) need more dimensions than instructed ones? Does EMG need fewer dimensions than pose within a stage?

✍️ **Q1.** Why do we fit each subspace on **train** frames and score on **test** frames, instead of
using all frames for both? What would happen to the diagonal of the next task's matrix (a stage scored with its
own subspace) if we didn't?

> *Your answer:*

---
## Task 2 · Cross-stage variance-explained ratio
The paper's main measure of sharing: **how much of stage A's variance does stage B's subspace capture, relative to
A's own subspace?**

$$\text{ratio}(A \leftarrow B) = \frac{\lVert X_A^c\, Q_B \rVert^2}{\lVert X_A^c\, Q_A \rVert^2}$$

`X_A^c` is A's test data centred on its own mean; `X @ Q` gives each frame's coordinates in the subspace, and the
sum of their squares is the variance that lies inside it (`captured`). A ratio of 1 means B's subspace is as good
for A as A's own; a ratio near what a **random** subspace gets means B tells you nothing about A.

**The null.** Even a random K-dimensional subspace captures some variance (roughly its share of the dimensions).
So for every pair we also compute the ratio with random orthonormal subspaces of the same size, averaged over 20
draws. A random orthonormal basis: QR-decompose a matrix of Gaussian random numbers and keep `Q`.

A ratio can be **above 1**: A's own subspace was fitted on A's *early* frames, and if A drifts, another stage's
subspace can fit A's *late* frames better.

In [ ]:
def captured(X: np.ndarray, Q: np.ndarray) -> float:
    """Variance of X (centred on its OWN mean) lying inside the subspace with orthonormal columns Q: ||Xc Q||^2."""
    # TODO
    raise NotImplementedError


def ratio(X_a: np.ndarray, Q_a: np.ndarray, Q_b: np.ndarray) -> float:
    """captured(X_a, Q_b) / captured(X_a, Q_a)."""
    # TODO
    raise NotImplementedError


def random_basis(D: int, d: int, rng: np.random.Generator) -> np.ndarray:
    """(D, d) random orthonormal columns: the Q of a QR decomposition of a (D, d) Gaussian matrix."""
    # TODO
    raise NotImplementedError


def ratio_matrix(X_test: dict[str, np.ndarray], Q: dict[str, np.ndarray]) -> np.ndarray:
    """(15, 15) M[a, b] = ratio(X_test[a], Q[a], Q[b]), rows and columns in sorted stage order. Diagonal = 1."""
    # TODO
    raise NotImplementedError


def null_ratio_matrix(X_test: dict[str, np.ndarray], Q: dict[str, np.ndarray], n_rand: int = 20,
                      seed: int = 0) -> np.ndarray:
    """(15, 15) the same ratio when B's subspace is replaced by random subspaces of the same size, averaged
    over n_rand draws. Draw the n_rand random bases once per column b (rng = np.random.default_rng(seed),
    columns in sorted order) and reuse them for every row a."""
    # TODO
    raise NotImplementedError


M_emg, M_pose = None, None      # TODO: ratio_matrix for EMG and for pose
N_emg, N_pose = None, None      # TODO: null_ratio_matrix for EMG and for pose (seed=SEED)

#### 💡 Stuck on Task 2? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `captured`: centre `X` with its own column means, project with `@ Q`, square everything, add it all up.
2. `ratio`: two calls to `captured`.
3. `random_basis`: a (D, d) matrix from `rng.normal`, then `np.linalg.qr`; keep the first output.
4. `ratio_matrix`: a double loop over sorted stage names, rows = a, columns = b.
5. `null_ratio_matrix`: loop over columns b; draw the random bases for b once; for each row a average the ratio
   over those bases.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.qr(A)` returns `(Q, R)`; for a tall (D, d) matrix `Q` is (D, d) with orthonormal columns.
- A list comprehension builds a matrix row by row: `np.array([[f(a, b) for b in names] for a in names])`.

```python
Q, _ = np.linalg.qr(np.random.default_rng(0).normal(size=(5, 2)))
np.round(Q.T @ Q, 6)     # identity: orthonormal columns
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def captured(X, Q):
    Xc = X - X.mean(axis=0)
    return float(np.sum((Xc @ ____) ** 2))

def ratio(X_a, Q_a, Q_b):
    return captured(X_a, ____) / captured(X_a, ____)

def random_basis(D, d, rng):
    Q, _ = np.linalg.qr(rng.normal(size=(____, ____)))
    return Q

def ratio_matrix(X_test, Q):
    names = sorted(X_test)
    return np.array([[ratio(X_test[a], Q[a], Q[b]) for b in names] for a in ____])

def null_ratio_matrix(X_test, Q, n_rand=20, seed=0):
    rng = np.random.default_rng(seed)
    names = sorted(X_test)
    D = Q[names[0]].shape[0]
    out = np.zeros((len(names), len(names)))
    for j, b in enumerate(names):
        rand = [random_basis(D, Q[b].shape[1], rng) for _ in range(n_rand)]
        for i, a in enumerate(names):
            out[i, j] = np.mean([ratio(X_test[a], Q[a], R) for R in ____])
    return out

M_emg, M_pose = ratio_matrix(X_emg, Q_emg), ratio_matrix(X_pose, Q_pose)
N_emg, N_pose = null_ratio_matrix(X_emg, Q_emg, seed=SEED), null_ratio_matrix(X_pose, Q_pose, seed=SEED)
```

</details>

In [ ]:
# ✅ Check Task 2
Xa = np.random.default_rng(1).normal(size=(200, 5)); Qa = ref.stage_basis(Xa, 2); Qb = ref.stage_basis(Xa[:, ::-1], 2)
assert abs(captured(Xa, Qa) - ref.captured(Xa, Qa)) < 1e-6, "captured: centre X on its own mean, then ||Xc Q||^2"
assert abs(ratio(Xa, Qa, Qb) - ref.ratio(Xa, Qa, Qb)) < 1e-9
R = random_basis(16, 4, np.random.default_rng(0))
assert R.shape == (16, 4) and np.allclose(R.T @ R, np.eye(4), atol=1e-8), "random_basis: orthonormal (16, 4)"
assert M_emg.shape == (15, 15) and np.allclose(np.diag(M_emg), 1), "diagonal of the ratio matrix must be 1"
assert np.allclose(M_emg, ref.ratio_matrix(X_emg, Q_emg), atol=1e-8), "EMG ratio matrix differs from the reference"
assert np.allclose(M_pose, ref.ratio_matrix(X_pose, Q_pose), atol=1e-8), "pose ratio matrix differs"
assert np.allclose(N_emg, ref.null_ratio_matrix(X_emg, Q_emg, seed=SEED), atol=1e-8), \
    "EMG null differs: draw the bases once per column, in sorted order, with default_rng(seed)"
off = ~np.eye(15, dtype=bool)
print(f"✅ mean off-diagonal ratio: EMG {M_emg[off].mean():.2f} (random {N_emg[off].mean():.2f}) · "
      f"pose {M_pose[off].mean():.2f} (random {N_pose[off].mean():.2f})")

In [ ]:
# 📊 Ratio matrices: row = the stage being explained (A), column = whose subspace is used (B)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
for a, M, title in [(ax[0], M_emg, "EMG"), (ax[1], M_pose, "pose")]:
    im = a.imshow(M, vmin=0, vmax=1.2, cmap="viridis")
    a.set_xticks(range(15), [s[:12] for s in STAGES], rotation=90, fontsize=7)
    a.set_yticks(range(15), [s[:12] for s in STAGES], fontsize=7)
    a.set_title(f"{title}: variance-explained ratio"); plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.show()

*What to look for:* is the EMG matrix uniformly bright (every stage's space works for every other), while the pose matrix has dark rows or columns (stages with private postures)?

✍️ **Q2.** A random 4-dimensional subspace already captures a sizeable share of the variance
(look at the null numbers). Why is that, and why does it make "the ratio is 0.8" meaningless without the null?

> *Your answer:*

---
## Task 3 · Principal angles between stage subspaces
The ratio asks "how much variance"; **principal angles** ask "how aligned are the two spaces themselves". For two
subspaces with orthonormal columns `Q_a` and `Q_b`, the singular values of `Q_a.T @ Q_b` are the cosines of the
principal angles. The first angle is the **smallest angle between any direction in A and any direction in B**:
0° = the spaces share a direction exactly, 90° = completely perpendicular.

We compare the **first** principal angle of every pair of stages with the angle between two **random** subspaces of
the same size. Shared structure shows up as angles much smaller than random.

In [ ]:
def principal_angles(Q_a: np.ndarray, Q_b: np.ndarray) -> np.ndarray:
    """Principal angles in DEGREES, smallest first.

    Parameters
    ----------
    Q_a : (D, d_a) orthonormal columns
    Q_b : (D, d_b) orthonormal columns

    Returns
    -------
    (min(d_a, d_b),) float in [0, 90]
    """
    # TODO
    raise NotImplementedError


pairs = [(a, b) for i, a in enumerate(STAGES) for j, b in enumerate(STAGES)
         if j > i and not SIMUL[i, j]]                                    # 104 pairs (simultaneous pair left out)
first_angle = {"EMG": [], "pose": [], "EMG random": [], "pose random": []}
rng_angles = np.random.default_rng(SEED)
for a, b in pairs:
    # TODO: append the FIRST principal angle between Q_emg[a] and Q_emg[b], and between Q_pose[a] and Q_pose[b]
    # TODO: the same for two random_basis subspaces (16 × K for "EMG random", 20 × K for "pose random"),
    #       drawn with rng_angles in this order: EMG random a, EMG random b, pose random a, pose random b
    ...

#### 💡 Stuck on Task 3? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Multiply `Q_a.T @ Q_b` → a small (d_a, d_b) matrix. Its singular values are cosines.
2. Clip them into [−1, 1] (rounding can push a value to 1.0000000002), take `arccos`, convert to degrees.
   Singular values come largest first, so angles come out smallest first.
3. In the loop, the first angle is element `[0]`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.svd(M, compute_uv=False)` → singular values only. `np.clip(x, -1, 1)`. `np.degrees(np.arccos(c))`.

```python
Qa = np.eye(3)[:, :1]; Qb = np.array([[1.0], [1.0], [0.0]]) / np.sqrt(2)
np.degrees(np.arccos(np.linalg.svd(Qa.T @ Qb, compute_uv=False)))   # [45.]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def principal_angles(Q_a, Q_b):
    s = np.linalg.svd(____.T @ ____, compute_uv=False)
    return np.degrees(np.arccos(np.clip(s, -1.0, 1.0)))

for a, b in pairs:
    first_angle["EMG"].append(principal_angles(Q_emg[a], Q_emg[b])[0])
    first_angle["pose"].append(principal_angles(____, ____)[0])
    first_angle["EMG random"].append(principal_angles(random_basis(16, K, rng_angles), random_basis(16, K, rng_angles))[0])
    first_angle["pose random"].append(principal_angles(random_basis(20, K, rng_angles), random_basis(____, K, rng_angles))[0])
```

</details>

In [ ]:
# ✅ Check Task 3
Qa = np.eye(3)[:, :1]; Qb = np.array([[1.0], [1.0], [0.0]]) / np.sqrt(2)
assert np.allclose(principal_angles(Qa, Qb), [45.0]), "two lines 45° apart should give [45.]"
assert np.allclose(principal_angles(Q_emg[STAGES[0]], Q_emg[STAGES[0]]), 0, atol=1e-4), "a space with itself: all 0°"
assert np.allclose(principal_angles(Q_emg[STAGES[0]], Q_emg[STAGES[1]]),
                   ref.principal_angles(Q_emg[STAGES[0]], Q_emg[STAGES[1]]), atol=1e-6)
assert all(len(v) == len(pairs) for v in first_angle.values()), f"one first angle per pair of stages ({len(pairs)} pairs)"
print("✅ median first principal angle: " + ", ".join(f"{k} {np.median(v):.1f}°" for k, v in first_angle.items()))

In [ ]:
# 📊 First principal angle between every pair of stages, against random subspaces
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(0, 90, 46)
for key, color in [("EMG", "C0"), ("pose", "C1"), ("EMG random", "0.6"), ("pose random", "0.3")]:
    ax.hist(first_angle[key], bins=bins, histtype="step", lw=2, color=color, label=key)
ax.set_xlabel("first principal angle (°)"); ax.set_ylabel("pairs of stages"); ax.legend(); plt.show()

*What to look for:* real angles piled up near 0° and random ones far to the right means the stages share at least one direction. Which signal sits closer to 0°?

✍️ **Q3.** The first principal angle between EMG subspaces is tiny for almost every pair of
stages. Think about what the **first** principal direction of an EMG envelope usually is (look at notebook 02's
first PCA component). Is a shared first direction strong evidence of shared *synergies*, or could something simpler
explain it?

> *Your answer:*

---
## Task 4 · Is EMG less shared across stages than pose? (the paper's prediction)
Lazzari & Saxena found muscle activity **less** shared across tasks than the network's activity, which here plays
the role of the movement (pose). We test whether the same holds for real EMG and pose.

The raw ratios can't be compared directly between EMG and pose: the random null differs (16 vs 20 dimensions). So
first rescale every ratio into a **sharing index**:

$$\text{index} = \frac{\text{ratio} - \text{null}}{1 - \text{null}}$$

0 = no better than a random subspace, 1 = as good as the stage's own subspace.

Then, per stage, average its row without the diagonal and without the simultaneous pair `SIMUL` (how well the
**other** stages' subspaces explain it), and
take the paired difference EMG − pose. For the confidence interval, average the stages within each independent
**group** first (14 groups), then bootstrap over groups with `protocol.session_bootstrap_ci`.

In [ ]:
def sharing_index(M: np.ndarray, M_null: np.ndarray) -> np.ndarray:
    """Element-wise (M - M_null) / (1 - M_null)."""
    # TODO
    raise NotImplementedError


def off_diagonal_row_means(M: np.ndarray, exclude: np.ndarray | None = None) -> np.ndarray:
    """(S,) the mean of each row of an (S, S) matrix, leaving out the diagonal entry and any entry where the
    (S, S) bool matrix `exclude` is True (pass exclude=SIMUL)."""
    # TODO
    raise NotImplementedError


def group_means(values: np.ndarray) -> np.ndarray:
    """(15,) per-stage values in STAGES order -> (14,) one mean per independent group (uses group_of_stage)."""
    g = np.array([group_of_stage[s] for s in STAGES])
    return np.array([values[g == i].mean() for i in range(len(groups))])


idx_emg, idx_pose = None, None   # TODO: (15,) off-diagonal row means (exclude=SIMUL) of the sharing index, EMG and pose
diff_ci = None                   # TODO: protocol.session_bootstrap_ci(group_means(idx_emg - idx_pose))  → (mean, low, high)

#### 💡 Stuck on Task 4? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `sharing_index` is one line of element-wise arithmetic.
2. `off_diagonal_row_means`: build a True/False mask that is False on the diagonal (and, if `exclude` is given,
   also False where `exclude` is True), then average each row over the True entries.
3. Compute the index matrices for EMG and pose, their off-diagonal row means, the per-stage difference, collapse to
   the 14 groups, and bootstrap.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `~np.eye(S, dtype=bool)` → True everywhere except the diagonal.
- `M[i, mask[i]]` → row i without its diagonal entry.

```python
M = np.array([[1.0, 0.4, 0.6], [0.2, 1.0, 0.8], [0.5, 0.5, 1.0]])
mask = ~np.eye(3, dtype=bool)
[M[i, mask[i]].mean() for i in range(3)]     # [0.5, 0.5, 0.5]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def sharing_index(M, M_null):
    return (M - ____) / (1.0 - ____)

def off_diagonal_row_means(M, exclude=None):
    mask = ~np.eye(len(M), dtype=bool)
    if exclude is not None:
        mask &= ~____
    return np.array([M[i, mask[i]].mean() for i in range(len(____))])

idx_emg  = off_diagonal_row_means(sharing_index(M_emg, N_emg), exclude=SIMUL)
idx_pose = off_diagonal_row_means(sharing_index(____, ____), exclude=SIMUL)
diff_ci  = protocol.session_bootstrap_ci(group_means(____ - ____))
```

</details>

In [ ]:
# ✅ Check Task 4
M_t = np.array([[1.0, 0.4, 0.6], [0.2, 1.0, 0.8], [0.5, 0.5, 1.0]])
assert np.allclose(off_diagonal_row_means(M_t), [0.5, 0.5, 0.5]), "leave out the diagonal"
assert np.allclose(sharing_index(np.array([0.5]), np.array([0.25])), [1 / 3]), "(0.5 - 0.25) / (1 - 0.25) = 1/3"
ex = np.zeros((3, 3), bool); ex[0, 2] = True
assert np.allclose(off_diagonal_row_means(M_t, exclude=ex), [0.4, 0.5, 0.5]), "entries where exclude is True are left out"
assert np.allclose(idx_emg, ref.off_diagonal_row_means(ref.sharing_index(M_emg, N_emg), exclude=SIMUL)), \
    "pass exclude=SIMUL so the simultaneous pair is left out"
assert np.allclose(idx_pose, ref.off_diagonal_row_means(ref.sharing_index(M_pose, N_pose), exclude=SIMUL))
r_ci = protocol.session_bootstrap_ci(group_means(idx_emg - idx_pose))
assert np.allclose(diff_ci, r_ci), "bootstrap the 14 GROUP means of the per-stage difference EMG − pose"
print(f"✅ sharing index: EMG {idx_emg.mean():.2f}, pose {idx_pose.mean():.2f}; "
      f"EMG − pose = {diff_ci[0]:+.2f} (95% CI {diff_ci[1]:+.2f} to {diff_ci[2]:+.2f}, 14 groups)")

In [ ]:
# 📊 Per-stage sharing index, EMG vs pose (one line per stage)
fig, ax = plt.subplots(figsize=(5, 5))
for e, p in zip(idx_emg, idx_pose):
    ax.plot([0, 1], [e, p], color="0.6", lw=1, marker="o")
ax.plot([0, 1], [idx_emg.mean(), idx_pose.mean()], color="k", lw=3, marker="s", label="mean")
ax.axhline(0, ls="--", color="gray"); ax.set_xticks([0, 1], ["EMG", "pose"]); ax.set_xlim(-0.3, 1.3)
ax.set_ylabel("sharing index (0 = random, 1 = own)"); ax.legend(); plt.show()

*What to look for:* do most lines slope the same way? A clear slope **down** from EMG to pose means EMG is *more* shared than pose: the **opposite** of the paper's prediction.

✍️ **Q4.** Did the paper's prediction (muscle activity less shared than movement) hold here?
Give two reasons why our setting could differ from theirs: think about what 16 skin electrodes record compared with
6 simulated muscles, and what the "other side" is in each case (a network's activity vs a hand's joint angles).

> *Your answer:*

---
## Task 5 · How aligned are the EMG and pose latents? (CCA)
The paper compares muscle and network latents with **canonical correlation analysis (CCA)**. CCA finds a direction in
the EMG latents and a direction in the pose latents whose time courses are **as correlated as possible**, then the
next most correlated pair at right angles to the first, and so on.

**From scratch, in three steps.** With centred `X` (T × p) and `Y` (T × q):
1. Covariances `Cxx = XᵀX/(T−1)`, `Cyy`, and the cross-covariance `Cxy = XᵀY/(T−1)`. Add a small ridge to `Cxx` and
   `Cyy` (`reg × mean variance × I`) so they can always be inverted.
2. **Whiten** both: `Kx = Cxx^(−1/2)`, `Ky = Cyy^(−1/2)` (from the eigendecomposition `V diag(1/√w) Vᵀ`). After
   whitening, every direction has variance 1, so covariance = correlation.
3. SVD of `Kx Cxy Ky = U diag(ρ) Vᵀ`: the singular values `ρ` are the canonical correlations, and the directions in
   the original coordinates are `Wx = Kx U`, `Wy = Ky V`.

**Score on held-out data.** In-sample canonical correlations are always optimistic. We fit on train frames and
report the correlation of each canonical pair on **test** frames (`cca_test_corr`).

**Inputs.** EMG side: the K = 4 synergy activations (NMF fitted on train EMG), log-transformed (EMG noise grows
with its size; the log evens that out). Pose side: the K = 4 pose PCA scores. **EMG leads movement**, so we also pair
the pose at time t with the activations from `lag` frames earlier (`lag_within`: 0, 100, 200 ms).

**Null:** the activations circularly shifted in time within each recording (`protocol.circular_shift_null`). Each
signal keeps its own structure, but their alignment is destroyed.

In [ ]:
def lag_within(X: np.ndarray, rec: np.ndarray, lag: int) -> np.ndarray:
    """Row t = X[t - lag] from the SAME recording; the first `lag` rows of each recording repeat its first frame.

    Parameters
    ----------
    X   : (N, D) float
    rec : (N,) int, recording of each row (rows of a recording are consecutive and in time order)
    lag : int ≥ 0, in frames

    Returns
    -------
    (N, D) float
    """
    # TODO
    raise NotImplementedError


def cca_fit(X: np.ndarray, Y: np.ndarray, reg: float = 1e-3
            ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Canonical correlation analysis (the three steps above).

    Parameters
    ----------
    X : (T, p) float;  Y : (T, q) float;  reg : ridge as a fraction of the mean variance

    Returns
    -------
    mean_x (p,), mean_y (q,), Wx (p, m), Wy (q, m), rho (m,) with m = min(p, q), largest correlation first
    """
    # TODO
    raise NotImplementedError


def cca_test_corr(X: np.ndarray, Y: np.ndarray, fit: tuple[np.ndarray, ...]) -> np.ndarray:
    """(m,) correlation of each canonical pair on NEW data X, Y, using the means and directions in `fit`."""
    # TODO
    raise NotImplementedError


# Synergy activations (NMF on train EMG) and pose PCA scores (valid train poses), for every frame
_, W_syn, _ = ref02.nmf_fit(emg[train], K, n_iter=200, seed=SEED)
log_c = np.log(ref02.nmf_transform(emg, W_syn, n_iter=200, seed=SEED) + 1e-3)   # (N, K)
p_mean, p_comps = ref02.pca_fit(pose[train & pose_valid], K)
z_pose = ref02.pca_transform(pose, p_mean, p_comps)                            # (N, K)
log_c_null = protocol.circular_shift_null(log_c, rec, np.random.default_rng(SEED))

LAGS = [0, 5, 10]                                  # frames: 0, 100, 200 ms
cca_test = {"real": {}, "null": {}}                # lag → (K,) test canonical correlations
fit_tr, fit_te = train & pose_valid, test & pose_valid
# TODO: for each lag: lag the activations (real and null) with lag_within, cca_fit on the fit_tr rows,
#       cca_test_corr on the fit_te rows; store in cca_test["real"][lag] and cca_test["null"][lag]

#### 💡 Stuck on Task 5? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `lag_within`: loop over recordings; for each, build the shifted block by putting `lag` copies of its first row in
   front and dropping its last `lag` rows.
2. `cca_fit`: centre; three covariance matrices; ridge; inverse square roots with `np.linalg.eigh`; SVD; map the
   directions back with `Kx @ U` and `Ky @ Vt.T`.
3. `cca_test_corr`: project the new data (centred with the TRAIN means) onto `Wx` and `Wy`, then the correlation of
   each pair of columns.
4. The loop: the same three calls for "real" and for "null".

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `w, V = np.linalg.eigh(C)` (for symmetric C); `V @ np.diag(1 / np.sqrt(w)) @ V.T` → C^(−1/2).
- Correlation of matching columns: centre each column, then `np.sum(U * V, 0) / np.sqrt(np.sum(U**2, 0) * np.sum(V**2, 0))`.
- `np.concatenate([np.repeat(x[:1], lag, axis=0), x[:len(x) - lag]])` shifts one block.

```python
x = np.arange(5)[:, None]
np.concatenate([np.repeat(x[:1], 2, axis=0), x[:3]]).ravel()   # [0, 0, 0, 1, 2]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def lag_within(X, rec, lag):
    out = X.copy()
    if lag == 0:
        return out
    for r in np.unique(rec):
        idx = np.where(rec == r)[0]
        x = X[idx]
        out[idx] = np.concatenate([np.repeat(x[:1], ____, axis=0), x[: len(idx) - ____]])
    return out

def cca_fit(X, Y, reg=1e-3):
    mx, my = X.mean(axis=0), Y.mean(axis=0)
    Xc, Yc = X - mx, Y - my
    n = len(X) - 1
    Cxx, Cyy, Cxy = Xc.T @ Xc / n, Yc.T @ Yc / n, Xc.T @ ____ / n
    Cxx = Cxx + reg * np.trace(Cxx) / len(Cxx) * np.eye(len(Cxx))
    Cyy = Cyy + reg * np.trace(Cyy) / len(Cyy) * np.eye(len(Cyy))
    def inv_sqrt(C):
        w, V = np.linalg.eigh(C)
        return V @ np.diag(1 / np.sqrt(np.maximum(w, 1e-12))) @ V.T
    Kx, Ky = inv_sqrt(Cxx), inv_sqrt(____)
    U, rho, Vt = np.linalg.svd(Kx @ Cxy @ Ky, full_matrices=False)
    return mx, my, Kx @ ____, Ky @ Vt.T, rho

def cca_test_corr(X, Y, fit):
    mx, my, Wx, Wy, _ = fit
    U, V = (X - mx) @ Wx, (Y - my) @ ____
    U, V = U - U.mean(axis=0), V - V.mean(axis=0)
    return np.sum(U * V, axis=0) / np.sqrt(np.sum(U ** 2, axis=0) * np.sum(V ** 2, axis=0))

for lag in LAGS:
    for key, src in [("real", log_c), ("null", log_c_null)]:
        Xl = lag_within(src, rec, lag)
        f = cca_fit(Xl[fit_tr], z_pose[fit_tr])
        cca_test[key][lag] = cca_test_corr(Xl[____], z_pose[____], f)
```

</details>

In [ ]:
# ✅ Check Task 5
x_t = np.arange(10.0)[:, None]; rec_t = np.repeat([0, 1], 5)
assert np.allclose(lag_within(x_t, rec_t, 2).ravel(), [0, 0, 0, 1, 2, 5, 5, 5, 6, 7]), \
    "lag_within must never borrow frames from the previous recording"
g = np.random.default_rng(3); Z = g.normal(size=(2000, 2))
Xs = np.c_[Z[:, 0] + 0.1 * g.normal(size=2000), g.normal(size=2000), g.normal(size=2000)]
Ys = np.c_[g.normal(size=2000), Z[:, 0] + 0.1 * g.normal(size=2000)]
f_t = cca_fit(Xs, Ys)
assert f_t[2].shape == (3, 2) and f_t[3].shape == (2, 2) and f_t[4].shape == (2,), "Wx (p, m), Wy (q, m), rho (m,)"
assert np.allclose(f_t[4], ref.cca_fit(Xs, Ys)[4], atol=1e-8), "canonical correlations differ from the reference"
assert f_t[4][0] > 0.98, "a shared signal should give a first canonical correlation near 1"
assert np.allclose(np.abs(cca_test_corr(Xs, Ys, f_t)), np.abs(ref.cca_test_corr(Xs, Ys, ref.cca_fit(Xs, Ys))), atol=1e-6)
for lag in LAGS:
    r_fit = ref.cca_fit(ref.lag_within(log_c, rec, lag)[fit_tr], z_pose[fit_tr])
    r_te = ref.cca_test_corr(ref.lag_within(log_c, rec, lag)[fit_te], z_pose[fit_te], r_fit)
    assert np.allclose(np.abs(cca_test["real"][lag]), np.abs(r_te), atol=1e-6), f"test correlations differ at lag {lag}"
print("✅ CCA works. First canonical correlation on TEST frames: " +
      ", ".join(f"{lag * 20} ms: real {cca_test['real'][lag][0]:.2f} / null {cca_test['null'][lag][0]:.2f}" for lag in LAGS))

In [ ]:
# 📊 Held-out canonical correlations, real vs time-shifted null, at each lag
fig, ax = plt.subplots(1, len(LAGS), figsize=(12, 3), sharey=True)
for a, lag in zip(ax, LAGS):
    a.bar(np.arange(K) - 0.2, cca_test["real"][lag], 0.4, label="real")
    a.bar(np.arange(K) + 0.2, cca_test["null"][lag], 0.4, color="0.6", label="shifted null")
    a.axhline(0, color="k", lw=0.5); a.set_title(f"EMG {lag * 20} ms before pose"); a.set_xlabel("canonical pair")
ax[0].set_ylabel("test correlation"); ax[0].legend(); plt.tight_layout(); plt.show()

*What to look for:* real bars clearly above the null bars, and whether they grow when the EMG is taken from **before** the pose (EMG leads movement).

✍️ **Q5.** The in-sample canonical correlations (`fit[4]`) are larger than the held-out ones.
Why? And why could the held-out correlation at lag 0 be close to the null even if synergies really drive the hand?
(Hint: notebook 04, and what EMG encodes: position, or force and change?)

> *Your answer:*

---
## Task 6 · Instructed → free-style: can learned primitives explain new combinations?
The paper's strongest form of composition is that a **new** task is built from primitives learned on other tasks.
Our closest analogue: fit synergies on the **instructed** stages only (pinches, counting, pointing …), then explain the
**free-style** stages (`FingerFreeform`, `OneHandedFreeStyle`, `TwoHandedFreeStyle`) with those synergies held fixed
(only the activations `C` are refit, `nmf_transform`).

Compare three sets of K = 4 synergies on each free-style stage's **test** frames:
- **borrowed:** fitted on the instructed stages' train frames;
- **own:** fitted on that free-style stage's own train frames (the ceiling);
- **random:** `ref04.random_W` (the floor).

Report **centred VAF** too (`vaf_centred`: the error is compared with the data minus its mean). Uncentered VAF
gives credit for rebuilding the average level, which any synergies can do (notebook 02).

*unconstrained* is left out of both sides: it is the **idle** hand recorded while the other hand did
*OneHandedFreeStyle*, so it is neither an instructed movement nor a deliberate free-style one.

In [ ]:
def vaf_centred(X: np.ndarray, X_hat: np.ndarray) -> float:
    """Centred VAF in %: 100 * (1 - ||X - X_hat||^2 / ||X - mean(X)||^2). 0 = no better than the mean."""
    # TODO
    raise NotImplementedError


FREE = [s for s in STAGES if ref.is_free_style(s)]          # the 3 free-style stages
IDLE = [s for s in STAGES if ref.is_idle(s)]                # "unconstrained": the idle hand (left out)
instructed = ~np.isin(stage_per_frame, FREE + IDLE)         # (N,) bool: the 11 instructed stages
W_random = ref.random_W(K, 16, seed=SEED)

transfer: dict[str, dict[str, tuple[float, float]]] = {}    # stage → {"borrowed"/"own"/"random": (VAF, centred VAF)}
# TODO: W_borrowed = synergies (ref02.nmf_fit, K, n_iter=300, seed=SEED) on emg[train & instructed]
# TODO: for each free-style stage s: W_own from emg[train & (stage_per_frame == s)] (same settings);
#       on X_te = emg[test & (stage_per_frame == s)], for each W: X_hat = nmf_transform(X_te, W, n_iter=300, seed=SEED) @ W;
#       transfer[s][name] = (ref02.vaf(X_te, X_hat), vaf_centred(X_te, X_hat))

#### 💡 Stuck on Task 6? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `vaf_centred` is notebook 02's VAF with the column means subtracted in the denominator.
2. Fit the borrowed synergies once. Then, per free-style stage: fit its own synergies, and score the three sets of
   synergies on its test frames.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `_, W, _ = ref02.nmf_fit(X, K, n_iter=300, seed=SEED)` keeps only the synergies.
- A dictionary of dictionaries: `transfer.setdefault(s, {})[name] = (a, b)`.

```python
X = np.array([[1.0, 2.0], [3.0, 4.0]])
np.sum((X - X.mean(axis=0)) ** 2)      # 4.0: the centred total
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def vaf_centred(X, X_hat):
    return float(100 * (1 - np.sum((X - X_hat) ** 2) / np.sum((X - X.mean(axis=____)) ** 2)))

_, W_borrowed, _ = ref02.nmf_fit(emg[train & ____], K, n_iter=300, seed=SEED)
for s in FREE:
    in_s = stage_per_frame == s
    _, W_own, _ = ref02.nmf_fit(emg[train & in_s], K, n_iter=300, seed=SEED)
    X_te = emg[test & ____]
    for name, W in [("borrowed", W_borrowed), ("own", W_own), ("random", W_random)]:
        X_hat = ref02.nmf_transform(X_te, W, n_iter=300, seed=SEED) @ W
        transfer.setdefault(s, {})[name] = (ref02.vaf(X_te, X_hat), vaf_centred(____, ____))
```

</details>

In [ ]:
# ✅ Check Task 6
X_t = np.array([[1.0, 2.0], [3.0, 4.0]])
assert abs(vaf_centred(X_t, X_t.mean(axis=0) + 0 * X_t)) < 1e-9, "rebuilding with the mean should give 0"
assert abs(vaf_centred(X_t, X_t) - 100) < 1e-9
assert sorted(transfer) == sorted(FREE) and all(set(v) == {"borrowed", "own", "random"} for v in transfer.values())
_, r_Wb, _ = ref02.nmf_fit(emg[train & instructed], K, n_iter=300, seed=SEED)
s_chk = FREE[0]; X_chk = emg[test & (stage_per_frame == s_chk)]
r_val = ref.vaf_centred(X_chk, ref02.nmf_transform(X_chk, r_Wb, n_iter=300, seed=SEED) @ r_Wb)
assert abs(transfer[s_chk]["borrowed"][1] - r_val) < 0.5, \
    f"borrowed centred VAF on {s_chk} is {transfer[s_chk]['borrowed'][1]:.1f}%, reference {r_val:.1f}%"
print("✅ Centred VAF (borrowed / own / random):")
for s in FREE:
    b, o, r = (transfer[s][k][1] for k in ("borrowed", "own", "random"))
    print(f"   {s:22s} {b:5.1f} / {o:5.1f} / {r:6.1f}   borrowed ÷ own = {b / o:.2f}")

In [ ]:
# 📊 Free-style stages explained by borrowed, own and random synergies (centred VAF)
fig, ax = plt.subplots(figsize=(9, 3.5))
x = np.arange(len(FREE))
for i, (name, color) in enumerate([("borrowed", "C0"), ("own", "C2"), ("random", "0.6")]):
    ax.bar(x + (i - 1) * 0.27, [transfer[s][name][1] for s in FREE], 0.27, color=color, label=name)
ax.axhline(0, color="k", lw=0.5); ax.set_xticks(x, FREE, fontsize=8); ax.set_ylabel("centred VAF (%)")
ax.legend(); plt.tight_layout(); plt.show()

*What to look for:* borrowed bars as tall as the own bars, and both far above random. If the borrowed bar is ever **taller** than own, ask why (own synergies are fitted on less data).

✍️ **Q6.** Instructed synergies explain free-style EMG about as well as the free-style
stage's own synergies. Is that evidence of **composition** (new movements built from old primitives), or would you
get the same result if every stage simply used the same few muscle patterns in the same proportions? What extra
analysis could tell the two apart? (Hint: compare the *activations* `C`, not just the reconstruction.)

> *Your answer:*

---
## ⚠️ Methodology: what this notebook can and can't tell you
Read this before writing the Result. Each row is a reason the numbers could look better (or different) than the truth, and what would test it.

| Issue | Why it matters | What would test it |
|---|---|---|
| **One person** | Every number describes one participant's stages | Repeat on a few users from the full dataset (stage 7) |
| **Simultaneous hands** | Left and right recordings of a block are one sample; OneHandedFreeStyle and unconstrained share blocks | Handled: CIs bootstrap 14 independent groups, and the simultaneous pair is left out of stage-vs-stage comparisons |
| **Stages differ in length and content** | TwoHandedFreeStyle has ~4× more frames; short stages give noisier subspaces | Subsample every stage to the same number of frames and recompute |
| **A shared first direction may be overall activation** | EMG envelopes rise and fall together (co-contraction, crosstalk), so every stage shares the 'everything louder' direction | Remove each frame's mean across channels (or the first PC) and recompute the sharing index |
| **Linear subspaces only** | PCA subspaces and CCA miss curved or switching structure | Nonlinear embeddings (research plan H1) or per-stage NMF instead of PCA |
| **Fixed K = 4 for both signals** | The EMG vs pose comparison depends on the subspace size | Repeat with K = 3 and 6 (the direction of the result should not change) |
| **Uncentered vs centred VAF** | Uncentered VAF gives free credit for the mean level | Task 6 reports both; judge transfer by the centred one |
| **Simulated vs real** | The paper's comparison is network activity vs simulated muscles; ours is EMG vs joint angles | Treat 'opposite to the paper' as a statement about our signals, not a refutation of the paper |

---
## Result · gate for stage 3
**Question:** Do the stages reuse shared structure, and is it shared beyond chance?

| Measure | Value (mean, 95% CI over sessions) | Baseline / null | Pass? |
|---|---|---|---|
| Sharing index, EMG (0 = random, 1 = own) | | | |
| Sharing index, pose | | | |
| EMG − pose sharing (paired, 14 groups) | | | |
| Median first principal angle, EMG / pose / random | | | |
| First held-out canonical correlation at 0 / 100 / 200 ms (real vs null) | | | |
| Free-style centred VAF: borrowed ÷ own (range over 3 stages) | | | |
| Free-style: borrowed vs random | | | |

**Gate:** sharing index clearly above 0 (CI excludes 0) **and** borrowed ÷ own centred VAF ≥ 0.9 on the free-style stages, far above random.

- **Pass →** stage 4 (notebook 04): the synergy space is shared, so ask what it encodes about movement (angle vs velocity, at what lag).
- **Fail →** the synergies are stage-specific: drop the single-bottleneck idea and fit per-stage models; report it as a finding.
- **Was your prediction right? Why or why not?**